In [27]:
import sys
sys.path.insert(0, '../../..')
from dependencies import *
from constants import *
from paths import *
import helper_functions
helper_functions.set_plot_style()

In [28]:
# ── LOAD ──────────────────────────────────────────────────────────────────────

TRIAL_WINDOW_SIZES   = [1, 2, 4, 8, 12, 16, 20, 24, 28, 32]        # in number of trials (~50s each)
TIME_WINDOW_SIZES_S  = [5, 10, 20, 30, 40]         # in seconds within a trial


# ── NH ────────────────────────────────────────────────────────────────────────

nh = {
    "envelope": {
        "trial": {
            n: {
                s: pd.read_csv(BEYOND_FUGLSANG_IN_SNHL_RESULTS / f"results_NH_trials_{n}_{s}_envelope.csv")
                for s in ("concatenate", "average")
            }
            for n in TRIAL_WINDOW_SIZES
        },

        "time": {
            n: {
                s: pd.read_csv(BEYOND_FUGLSANG_IN_SNHL_RESULTS / f"results_NH_time_{n}s_{s}_envelope.csv")
                for s in ("concatenate",)
            }
            for n in TIME_WINDOW_SIZES_S
        }
    },
    "envelope_onset": {
        "trial": {
            n: {
                s: pd.read_csv(BEYOND_FUGLSANG_IN_SNHL_RESULTS / f"results_NH_trials_{n}_{s}_envelope_onset.csv")
                for s in ("concatenate", "average")
            }
            for n in TRIAL_WINDOW_SIZES
        },

        "time": {
            n: {
                s: pd.read_csv(BEYOND_FUGLSANG_IN_SNHL_RESULTS / f"results_NH_time_{n}s_{s}_envelope_onset.csv")
                for s in ("concatenate",)
            }
            for n in TIME_WINDOW_SIZES_S
        }
    
    }
}

# ── HI ────────────────────────────────────────────────────────────────────────

hi = {
    "envelope" : {
        "trial": {
            n: {
                s: pd.read_csv(BEYOND_FUGLSANG_IN_SNHL_RESULTS / f"results_HI_trials_{n}_{s}_envelope.csv")
                for s in ("concatenate", "average")
            }
            for n in TRIAL_WINDOW_SIZES
        },

        "time": {
            n: {
                s: pd.read_csv(BEYOND_FUGLSANG_IN_SNHL_RESULTS / f"results_HI_time_{n}s_{s}_envelope.csv")
                for s in ("concatenate",)
            }
            for n in TIME_WINDOW_SIZES_S
        }
    },

    "envelope_onset": {
        "trial": {
            n: {
                s: pd.read_csv(BEYOND_FUGLSANG_IN_SNHL_RESULTS / f"results_HI_trials_{n}_{s}_envelope_onset.csv")
                for s in ("concatenate", "average")
            }
            for n in TRIAL_WINDOW_SIZES
        },

        "time": {
            n: {
                s: pd.read_csv(BEYOND_FUGLSANG_IN_SNHL_RESULTS / f"results_HI_time_{n}s_{s}_envelope_onset.csv")
                for s in ("concatenate",)
            }
            for n in TIME_WINDOW_SIZES_S
        }

    }
        
}

In [29]:
def run_all_stats(nh, hi):
    """
    Runs binomial tests (vs chance) and McNemar tests (envelope vs onset)
    for all window sizes and strategies.

    Returns a tidy DataFrame with one row per condition.
    """
    rows = []

    window_types = {
        "trial": TRIAL_WINDOW_SIZES,
        "time":  TIME_WINDOW_SIZES_S,
    }
    strategies_by_type = {
        "trial": ("concatenate", "average"),
        "time":  ("concatenate",),
    }
    groups = {"NH": nh, "HI": hi}

    # ── Binomial tests ────────────────────────────────────────────────────────
    for group_name, group_data in groups.items():
        for predictor in ("envelope", "envelope_onset"):
            for win_type, sizes in window_types.items():
                for size in sizes:
                    for strategy in strategies_by_type[win_type]:
                        df = group_data[predictor][win_type][size][strategy]

                        n_correct = int(df["correct"].sum())
                        n_total   = len(df)

                        binom = helper_functions.binomial_test(n_correct, n_total)
                        rows.append({
                            "group":     group_name,
                            "predictor": predictor,
                            "win_type":  win_type,
                            "size":      size,
                            "strategy":  strategy,
                            "test":      "binomial",
                            "n_correct": n_correct,
                            "n_total":   n_total,
                            "accuracy":  binom["accuracy"],
                            "p":         binom["p"],
                            "detail":    None,
                        })

    # ── McNemar: envelope vs onset (within group, window, strategy) ───────────
    for group_name, group_data in groups.items():
        for win_type, sizes in window_types.items():
            for size in sizes:
                for strategy in strategies_by_type[win_type]:
                    df_env   = group_data["envelope"][win_type][size][strategy]
                    df_onset = group_data["envelope_onset"][win_type][size][strategy]

                    merged = pd.merge(
                        df_env[["subject", "correct"]].rename(columns={"correct": "correct_env"}).reset_index(drop=True),
                        df_onset[["subject", "correct"]].rename(columns={"correct": "correct_onset"}).reset_index(drop=True),
                        left_index=True, right_index=True
                    )
                    mc = helper_functions.mcnemar_test(merged["correct_env"], merged["correct_onset"])
                    rows.append({
                        "group":     group_name,
                        "predictor": "envelope_vs_onset",
                        "win_type":  win_type,
                        "size":      size,
                        "strategy":  strategy,
                        "test":      "mcnemar",
                        "n_correct": None,
                        "n_total":   None,
                        "accuracy":  None,
                        "p":         mc["p"],
                        "detail":    mc["test_used"],
                    })

    return pd.DataFrame(rows)


stats_df = run_all_stats(nh, hi)
stats_df.to_csv(BEYOND_FUGLSANG_IN_SNHL_RESULTS / "all_stats.csv", index=False)

In [30]:
stats_df

,group,predictor,win_type,size,strategy,test,n_correct,n_total,accuracy,p,detail
0,NH,envelope,trial,1,concatenate,binomial,505.0,697.0,0.724534,1.242957e-33,NaN
1,NH,envelope,trial,1,average,binomial,505.0,697.0,0.724534,1.242957e-33,NaN
2,NH,envelope,trial,2,concatenate,binomial,282.0,348.0,0.810345,3.175500e-33,NaN
3,NH,envelope,trial,2,average,binomial,284.0,348.0,0.816092,1.676265e-34,NaN
4,NH,envelope,trial,4,concatenate,binomial,146.0,174.0,0.839080,9.231831e-21,NaN
...,...,...,...,...,...,...,...,...,...,...,...
145,HI,envelope_vs_onset,time,5,concatenate,mcnemar,NaN,NaN,NaN,3.209322e-03,chi-squared approximation (n_discordant=1198)
146,HI,envelope_vs_onset,time,10,concatenate,mcnemar,NaN,NaN,NaN,7.863173e-05,chi-squared approximation (n_discordant=616)
147,HI,envelope_vs_onset,time,20,concatenate,mcnemar,NaN,NaN,NaN,2.059032e-01,chi-squared approximation (n_discordant=250)
148,HI,envelope_vs_onset,time,30,concatenate,mcnemar,NaN,NaN,NaN,3.843055e-03,chi-squared approximation (n_discordant=115)


In [32]:
key_windows = [
    ("trial", 32, "concatenate"),
    ("trial",  1, "concatenate"),
    ("time",   5, "concatenate"),
    ("time",  30, "concatenate"),
]

for group in ("NH", "HI"):
    for predictor in ("envelope", "envelope_onset"):
        for win_type, size, strategy in key_windows:
            row = stats_df[
                (stats_df["group"]     == group) &
                (stats_df["predictor"] == predictor) &
                (stats_df["win_type"]  == win_type) &
                (stats_df["size"]      == size) &
                (stats_df["strategy"]  == strategy) &
                (stats_df["test"]      == "binomial")
            ]
            if row.empty:
                print(f"MISSING: {group} | {predictor} | {win_type} | {size} | {strategy}")
            else:
                print(row[["group", "predictor", "win_type", "size", "accuracy", "p"]].to_string(index=False))

group predictor win_type  size  accuracy       p
   NH  envelope    trial    32  0.952381 0.00001
group predictor win_type  size  accuracy            p
   NH  envelope    trial     1  0.724534 1.242957e-33
group predictor win_type  size  accuracy            p
   NH  envelope     time     5  0.591302 5.976783e-52
group predictor win_type  size  accuracy            p
   NH  envelope     time    30  0.705882 2.048355e-28
group      predictor win_type  size  accuracy        p
   NH envelope_onset    trial    32  0.809524 0.003599
group      predictor win_type  size  accuracy            p
   NH envelope_onset    trial     1  0.691535 9.571331e-25
group      predictor win_type  size  accuracy            p
   NH envelope_onset     time     5  0.569483 7.391613e-31
group      predictor win_type  size  accuracy            p
   NH envelope_onset     time    30   0.64132 3.904757e-14
group predictor win_type  size  accuracy            p
   HI  envelope    trial    32       1.0 2.384186e-07
group 